# Titanic survival — exploratory tree experiments

Execution checked on 2026-10-16 with the versions in `requirements.txt`, including both CV searches. Source outputs are cleared for a compact repository. Results are exploratory, not an independent final evaluation. See [Project Documentation](Project%20Documentation/README.md) for methodology and limitations.

The stratified 80/20 split (`random_state=42`) precedes removal of missing `Embarked` rows, leaving 710 training and 179 evaluation rows. The analysis notebook uses a different preparation order and split. `X_test` is repeatedly reused for model comparisons; age imputation, ticket frequencies and encoding are prepared before CV rather than separately within each fold. Full-dataset EDA also precedes splitting.

**Code assistance:** The code was developed with assistance from **CODEX**.


# titanic_survival_prediction ML model.

# libraries for the model

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv("Data/Titanic_dataset.csv")
df

In [ ]:
df["Pclass"].info()
df["Pclass"].describe()
df["Pclass"].value_counts(dropna=False)

In [ ]:
pd.crosstab(df["Pclass"], df["Survived"])

In [ ]:
pd.crosstab(
    df["Pclass"],
    df["Survived"],
    normalize="index"
) * 100

In [ ]:
df["Name"].info()
df["Name"].describe()
df["Name"].value_counts(dropna=False)

In [ ]:
df["Sex"].info()
df["Sex"].describe()
df["Sex"].value_counts(dropna=False)

In [ ]:
df["Age"].info()
df["Age"].describe()
df["Age"].value_counts(dropna=False)

In [ ]:
df["SibSp"].info()
df["SibSp"].describe()
df["SibSp"].value_counts(dropna=False)

In [ ]:
df["Parch"].info()
df["Parch"].describe()
df["Parch"].value_counts(dropna=False)

In [ ]:
df["Ticket"].info()
df["Ticket"].describe()
df["Ticket"].value_counts(dropna=False)

In [ ]:
df["Fare"].info()
df["Fare"].describe()
df["Fare"].value_counts(dropna=False)

In [ ]:
fare_counts = df["Fare"].value_counts().sort_index()

fare_counts

In [ ]:
import matplotlib.pyplot as plt

fare_counts = df["Fare"].value_counts().sort_index()

plt.scatter(fare_counts.index, fare_counts.values)
plt.xlabel("Fare")
plt.ylabel("Number of passengers")
plt.title("Fare vs Number of Passengers")
plt.show()

In [ ]:
df["Cabin"].info()
df["Cabin"].describe()
df["Cabin"].value_counts(dropna=False)

In [ ]:
df["Embarked"].info()
df["Embarked"].describe()
df["Embarked"].value_counts(dropna=False)

In [ ]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")

In [ ]:
df[["Name", "Title"]].head(10)

In [ ]:
df["Title"].info()
df["Title"].value_counts(dropna=False)

In [ ]:
pd.crosstab(df["Title"], df["Survived"])

In [ ]:
pd.crosstab(
    df["Title"],
    df["Survived"],
    normalize="index"
) * 100

In [ ]:
print("Media:", df["Age"].mean())
print("Mediana:", df["Age"].median())
print("Moda:", df["Age"].mode()[0])

In [ ]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
# ==========================================
# DATA PREPARATION & FEATURE ENGINEERING
# ==========================================

# 1. Drop PassengerId
X_train = X_train.drop(columns=["PassengerId"])
X_test = X_test.drop(columns=["PassengerId"])


# 2. Name / Title
# Normalize equivalent titles
title_mapping = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
}

X_train["Title"] = X_train["Title"].replace(title_mapping)
X_test["Title"] = X_test["Title"].replace(title_mapping)

# Group uncommon titles as Rare
common_titles = ["Mr", "Mrs", "Miss", "Master"]

X_train["Title"] = X_train["Title"].where(
    X_train["Title"].isin(common_titles),
    "Rare"
)

X_test["Title"] = X_test["Title"].where(
    X_test["Title"].isin(common_titles),
    "Rare"
)

# Drop original Name column
X_train = X_train.drop(columns=["Name"])
X_test = X_test.drop(columns=["Name"])


# 3. Age
# Learn median ONLY from training data
age_median = X_train["Age"].median()

X_train["Age"] = X_train["Age"].fillna(age_median)
X_test["Age"] = X_test["Age"].fillna(age_median)


# 4. Ticket
# Learn ticket frequencies from training data only
ticket_group_size = X_train["Ticket"].value_counts()

X_train["TicketGroupSize"] = (
    X_train["Ticket"]
    .map(ticket_group_size)
    .fillna(1)
)

X_test["TicketGroupSize"] = (
    X_test["Ticket"]
    .map(ticket_group_size)
    .fillna(1)
)

# Drop original Ticket
X_train = X_train.drop(columns=["Ticket"])
X_test = X_test.drop(columns=["Ticket"])


# 5. Cabin
X_train = X_train.drop(columns=["Cabin"])
X_test = X_test.drop(columns=["Cabin"])


# 6. Embarked
# Remove rows with missing Embarked from training data
train_mask = X_train["Embarked"].notna()

X_train = X_train.loc[train_mask]
y_train = y_train.loc[train_mask]

# Remove rows with missing Embarked from test data
test_mask = X_test["Embarked"].notna()

X_test = X_test.loc[test_mask]
y_test = y_test.loc[test_mask]


# 7. One-Hot Encoding
categorical_columns = ["Sex", "Embarked", "Title"]

X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    drop_first=False,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    drop_first=False,
    dtype=int
)


# 8. Ensure train and test have exactly the same columns
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nMissing values in train:")
print(X_train.isnull().sum())

print("\nMissing values in test:")
print(X_test.isnull().sum())

print("\nTrain columns:")
print(X_train.columns.tolist())

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [ ]:
# ==========================================
# DECISION TREE - MODEL TRAINING
# ==========================================

decision_tree = DecisionTreeClassifier(
    random_state=42
)

decision_tree.fit(X_train, y_train)

# Predictions
y_pred_dt = decision_tree.predict(X_test)

# Probabilities for ROC-AUC
y_prob_dt = decision_tree.predict_proba(X_test)[:, 1]


# ==========================================
# MODEL EVALUATION
# ==========================================

accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test, y_prob_dt)
conf_matrix_dt = confusion_matrix(y_test, y_pred_dt)


print("DECISION TREE RESULTS")
print("---------------------")
print(f"Accuracy:  {accuracy_dt:.4f}")
print(f"Precision: {precision_dt:.4f}")
print(f"Recall:    {recall_dt:.4f}")
print(f"F1-score:  {f1_dt:.4f}")
print(f"ROC-AUC:   {roc_auc_dt:.4f}")

print("\nConfusion Matrix:")
print(conf_matrix_dt)

In [ ]:
# ==========================================
# OVERFITTING CHECK - DECISION TREE
# ==========================================

# Predictions on training data
y_train_pred_dt = decision_tree.predict(X_train)

# Accuracy
train_accuracy_dt = accuracy_score(y_train, y_train_pred_dt)
test_accuracy_dt = accuracy_score(y_test, y_pred_dt)

print("DECISION TREE - TRAIN VS TEST")
print("-----------------------------")
print(f"Train Accuracy: {train_accuracy_dt:.4f}")
print(f"Test Accuracy:  {test_accuracy_dt:.4f}")
print(f"Difference:     {train_accuracy_dt - test_accuracy_dt:.4f}")

In [ ]:
# ==========================================
# REGULARIZED DECISION TREE - MAX DEPTH = 5
# ==========================================

decision_tree_depth5 = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

# Train model
decision_tree_depth5.fit(X_train, y_train)

# Predictions
y_pred_dt_depth5 = decision_tree_depth5.predict(X_test)
y_prob_dt_depth5 = decision_tree_depth5.predict_proba(X_test)[:, 1]

# Training predictions
y_train_pred_dt_depth5 = decision_tree_depth5.predict(X_train)


# ==========================================
# EVALUATION
# ==========================================

accuracy_dt_depth5 = accuracy_score(y_test, y_pred_dt_depth5)
precision_dt_depth5 = precision_score(y_test, y_pred_dt_depth5)
recall_dt_depth5 = recall_score(y_test, y_pred_dt_depth5)
f1_dt_depth5 = f1_score(y_test, y_pred_dt_depth5)
roc_auc_dt_depth5 = roc_auc_score(y_test, y_prob_dt_depth5)

conf_matrix_dt_depth5 = confusion_matrix(y_test, y_pred_dt_depth5)

train_accuracy_dt_depth5 = accuracy_score(
    y_train,
    y_train_pred_dt_depth5
)

gap_dt_depth5 = train_accuracy_dt_depth5 - accuracy_dt_depth5


print("REGULARIZED DECISION TREE - MAX_DEPTH = 5")
print("------------------------------------------")

print(f"Train Accuracy: {train_accuracy_dt_depth5:.4f}")
print(f"Test Accuracy:  {accuracy_dt_depth5:.4f}")
print(f"Train-Test Gap: {gap_dt_depth5:.4f}")

print("\nTEST METRICS")
print(f"Accuracy:  {accuracy_dt_depth5:.4f}")
print(f"Precision: {precision_dt_depth5:.4f}")
print(f"Recall:    {recall_dt_depth5:.4f}")
print(f"F1-score:  {f1_dt_depth5:.4f}")
print(f"ROC-AUC:   {roc_auc_dt_depth5:.4f}")

print("\nConfusion Matrix:")
print(conf_matrix_dt_depth5)

In [ ]:
# ==========================================
# DECISION TREE - MAX DEPTH COMPARISON
# ==========================================

results_depth = []

for depth in range(3, 11):

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    # Train
    model.fit(X_train, y_train)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    y_test_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)
    roc_auc = roc_auc_score(y_test, y_test_prob)

    results_depth.append({
        "Max Depth": depth,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Gap": train_acc - test_acc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc
    })


depth_results_df = pd.DataFrame(results_depth)

depth_results_df.round(4)

In [ ]:
# ==========================================
# RANDOM FOREST - BASELINE
# ==========================================

random_forest = RandomForestClassifier(
    random_state=42
)

# Train
random_forest.fit(X_train, y_train)

# Predictions
y_pred_rf = random_forest.predict(X_test)
y_prob_rf = random_forest.predict_proba(X_test)[:, 1]

# Train predictions
y_train_pred_rf = random_forest.predict(X_train)


# ==========================================
# EVALUATION
# ==========================================

train_accuracy_rf = accuracy_score(y_train, y_train_pred_rf)
test_accuracy_rf = accuracy_score(y_test, y_pred_rf)

precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)


print("RANDOM FOREST - BASELINE")
print("------------------------")

print(f"Train Accuracy: {train_accuracy_rf:.4f}")
print(f"Test Accuracy:  {test_accuracy_rf:.4f}")
print(f"Train-Test Gap: {train_accuracy_rf - test_accuracy_rf:.4f}")

print("\nTEST METRICS")
print(f"Accuracy:  {test_accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall:    {recall_rf:.4f}")
print(f"F1-score:  {f1_rf:.4f}")
print(f"ROC-AUC:   {roc_auc_rf:.4f}")

print("\nConfusion Matrix:")
print(conf_matrix_rf)

In [ ]:
 # ==========================================
# RANDOM FOREST - MAX DEPTH COMPARISON
# ==========================================

results_rf_depth = []

for depth in range(3, 11):

    model = RandomForestClassifier(
        max_depth=depth,
        random_state=42
    )

    # Train
    model.fit(X_train, y_train)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    y_test_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)
    roc_auc = roc_auc_score(y_test, y_test_prob)

    results_rf_depth.append({
        "Max Depth": depth,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Gap": train_acc - test_acc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc
    })


rf_depth_results_df = pd.DataFrame(results_rf_depth)

rf_depth_results_df.round(4)

In [ ]:
# ==========================================
# RANDOM FOREST - SPLIT / LEAF COMPARISON
# max_depth = 5
# ==========================================

results_rf_tuning = []

for min_split in [2, 5, 10, 20]:
    for min_leaf in [1, 2, 4, 8]:

        model = RandomForestClassifier(
            max_depth=5,
            min_samples_split=min_split,
            min_samples_leaf=min_leaf,
            random_state=42
        )

        # Train
        model.fit(X_train, y_train)

        # Predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        y_test_prob = model.predict_proba(X_test)[:, 1]

        # Metrics
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)
        precision = precision_score(y_test, y_test_pred)
        recall = recall_score(y_test, y_test_pred)
        f1 = f1_score(y_test, y_test_pred)
        roc_auc = roc_auc_score(y_test, y_test_prob)

        results_rf_tuning.append({
            "Min Split": min_split,
            "Min Leaf": min_leaf,
            "Train Accuracy": train_acc,
            "Test Accuracy": test_acc,
            "Gap": train_acc - test_acc,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "ROC-AUC": roc_auc
        })


rf_tuning_df = pd.DataFrame(results_rf_tuning)

rf_tuning_df.round(4)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# ==========================================
# RANDOM FOREST - GRID SEARCH CV
# ==========================================

rf_grid_model = RandomForestClassifier(
    random_state=42
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5, 6, 7],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4]
}

grid_search_rf = GridSearchCV(
    estimator=rf_grid_model,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

In [ ]:
print("Best Parameters:")
print(grid_search_rf.best_params_)

print("\nBest Cross-Validation F1:")
print(round(grid_search_rf.best_score_, 4))

In [ ]:
# ==========================================
# SELECTED RANDOM FOREST - EXPLORATORY EVALUATION
# ==========================================

best_rf = grid_search_rf.best_estimator_

# Test predictions
y_pred_best_rf = best_rf.predict(X_test)
y_prob_best_rf = best_rf.predict_proba(X_test)[:, 1]

# Train predictions
y_train_pred_best_rf = best_rf.predict(X_train)

# Metrics
train_accuracy_best_rf = accuracy_score(y_train, y_train_pred_best_rf)
test_accuracy_best_rf = accuracy_score(y_test, y_pred_best_rf)

precision_best_rf = precision_score(y_test, y_pred_best_rf)
recall_best_rf = recall_score(y_test, y_pred_best_rf)
f1_best_rf = f1_score(y_test, y_pred_best_rf)
roc_auc_best_rf = roc_auc_score(y_test, y_prob_best_rf)

conf_matrix_best_rf = confusion_matrix(y_test, y_pred_best_rf)

print("SELECTED RANDOM FOREST - EXPLORATORY")
print("-------------------")

print(f"Train Accuracy: {train_accuracy_best_rf:.4f}")
print(f"Test Accuracy:  {test_accuracy_best_rf:.4f}")
print(f"Train-Test Gap: {train_accuracy_best_rf - test_accuracy_best_rf:.4f}")

print("\nTEST METRICS")
print(f"Accuracy:  {test_accuracy_best_rf:.4f}")
print(f"Precision: {precision_best_rf:.4f}")
print(f"Recall:    {recall_best_rf:.4f}")
print(f"F1-score:  {f1_best_rf:.4f}")
print(f"ROC-AUC:   {roc_auc_best_rf:.4f}")

print("\nConfusion Matrix:")
print(conf_matrix_best_rf)

In [ ]:
# ==========================================
# DECISION TREE - GRID SEARCH CV
# ==========================================

dt_grid_model = DecisionTreeClassifier(
    random_state=42
)

param_grid_dt = {
    "max_depth": [2, 3, 4, 5, 6, 7, 8, 10],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "criterion": ["gini", "entropy"]
}

grid_search_dt = GridSearchCV(
    estimator=dt_grid_model,
    param_grid=param_grid_dt,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_search_dt.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search_dt.best_params_)

print("\nBest Cross-Validation F1:")
print(round(grid_search_dt.best_score_, 4))

In [ ]:
# ==========================================
# SELECTED DECISION TREE - EXPLORATORY EVALUATION
# ==========================================

best_dt = grid_search_dt.best_estimator_

# Predictions
y_pred_best_dt = best_dt.predict(X_test)
y_prob_best_dt = best_dt.predict_proba(X_test)[:, 1]
y_train_pred_best_dt = best_dt.predict(X_train)

# Metrics
train_accuracy_best_dt = accuracy_score(y_train, y_train_pred_best_dt)
test_accuracy_best_dt = accuracy_score(y_test, y_pred_best_dt)

precision_best_dt = precision_score(y_test, y_pred_best_dt)
recall_best_dt = recall_score(y_test, y_pred_best_dt)
f1_best_dt = f1_score(y_test, y_pred_best_dt)
roc_auc_best_dt = roc_auc_score(y_test, y_prob_best_dt)

conf_matrix_best_dt = confusion_matrix(y_test, y_pred_best_dt)

print("SELECTED DECISION TREE - EXPLORATORY")
print("-------------------")

print(f"Train Accuracy: {train_accuracy_best_dt:.4f}")
print(f"Test Accuracy:  {test_accuracy_best_dt:.4f}")
print(f"Train-Test Gap: {train_accuracy_best_dt - test_accuracy_best_dt:.4f}")

print("\nTEST METRICS")
print(f"Accuracy:  {test_accuracy_best_dt:.4f}")
print(f"Precision: {precision_best_dt:.4f}")
print(f"Recall:    {recall_best_dt:.4f}")
print(f"F1-score:  {f1_best_dt:.4f}")
print(f"ROC-AUC:   {roc_auc_best_dt:.4f}")

print("\nConfusion Matrix:")
print(conf_matrix_best_dt)